# DBSCAN (Density-Based Spatial Clustering of Applications with Noise)

**DBSCAN** is a powerful, non-parametric, density-based unsupervised clustering algorithm. Unlike K-Means, which partition data space into spherical regions around centroids, DBSCAN identifies clusters as continuous areas of high point density separated by areas of low point density. This allows it to discover clusters of arbitrary shapes and naturally detect noise (outliers) without pre-specifying the number of clusters.

In these detailed notes, we will cover:
1. **Core Concept:** Epsilon-neighborhood, Core/Border/Noise points, and density reachability/connectivity.
2. **Algorithmic Steps:** Step-by-step cluster discovery and expansion.
3. **Choosing Key Hyperparameters:** Mathematical heuristics for selecting `eps` ($\epsilon$) (using the $k$-distance elbow plot) and `min_samples`.
4. **Pros, Cons, & Failure Cases:** Performance on varying densities and high-dimensional spaces.
5. **Practical Python Demos:**
   - **Demo A:** Custom implementation of `DBSCANScratch` from scratch, tested on a non-linear dataset (moons) where K-Means fails.
   - **Demo B:** Grid visualization showing the effect of `eps` and `min_samples` on cluster boundaries.
   - **Demo C:** Automatically choosing `eps` using Scikit-Learn's `NearestNeighbors` and validating clustering quality.
   - **Summary Checklist** for DBSCAN.

## 1. Core Concepts & Point Classifications

DBSCAN defines clusters based on the density of points in a local region. It uses two central parameters: $\epsilon$ (`eps`) and `min_samples`.

### Core Definitions
- **$\epsilon$-Neighborhood ($N_\epsilon(x)$):** The set of all points within a Euclidean distance of $\epsilon$ from point $x$:
  $$N_\epsilon(x) = \{ y \in X \mid \text{dist}(x, y) \le \epsilon \}$$

Using these neighborhoods, DBSCAN classifies every point in the dataset into one of three categories:

1. **Core Points:** A point $x$ is a Core point if its $\epsilon$-neighborhood contains at least `min_samples` points (including $x$ itself):
   $$|N_\epsilon(x)| \ge \text{min\_samples}$$
2. **Border Points:** A point $y$ is a Border point if it is not a Core point, but it lies within the $\epsilon$-neighborhood of at least one Core point.
3. **Noise/Outlier Points:** A point $z$ is a Noise point if it is neither a Core point nor a Border point (i.e., it has fewer than `min_samples` in its neighborhood and contains no Core points in its neighborhood).

| Point Type | Condition | Part of Cluster? |
| :--- | :--- | :--- |
| **Core** | $|N_\epsilon(x)| \ge \text{min\_samples}$ | Yes (starts/expands clusters) |
| **Border** | $|N_\epsilon(x)| < \text{min\_samples}$ and $\exists c \in N_\epsilon(x)$ such that $c$ is a Core point | Yes (lies on the boundary of a cluster) |
| **Noise** | Otherwise | No (labeled as $-1$) |

---

### Density Connectivity
To understand how clusters are built, we define three relations between points:
- **Directly Density-Reachable:** A point $p$ is directly density-reachable from a point $q$ if $p \in N_\epsilon(q)$ and $q$ is a Core point.
- **Density-Reachable:** A point $p$ is density-reachable from $q$ if there is a chain of points $p_1, p_2, \dots, p_m$ (with $p_1 = q$ and $p_m = p$) where each $p_{i+1}$ is directly density-reachable from $p_i$.
- **Density-Connected:** Two points $p$ and $q$ are density-connected if there exists an intermediate point $o$ such that both $p$ and $q$ are density-reachable from $o$. DBSCAN clusters are defined as maximal sets of density-connected points.

## 2. The DBSCAN Algorithm

Given parameters $\epsilon$ and `min_samples`, DBSCAN proceeds as follows:

1. **Initialize Labels:** Mark all points as unvisited.
2. **Iterate Through Points:** For each unvisited point $x_i$:
   - Mark $x_i$ as visited.
   - Find its neighbors $N_\epsilon(x_i)$.
   - If $|N_\epsilon(x_i)| < \text{min\_samples}$:
     - Label $x_i$ as **Noise** (temporary label; it might be reassigned to a cluster later as a border point).
   - If $|N_\epsilon(x_i)| \ge \text{min\_samples}$:
     - Create a **new cluster** with $x_i$ as a Core point.
     - Expand the cluster using a queue structure. For each neighbor $y \in N_\epsilon(x_i)$:
       - If $y$ was labeled as Noise, change its label to the current cluster (it becomes a Border point).
       - If $y$ is unvisited:
         - Mark $y$ as visited and assign it to the current cluster.
         - Find its neighbors $N_\epsilon(y)$. If $|N_\epsilon(y)| \ge \text{min\_samples}$, add all of these neighbors to the queue to expand further.
3. **Complete:** Continue until all points in the dataset have been visited.

## 3. Selecting Hyperparameters & Key Limitations

### Choosing `min_samples`
- A common rule of thumb is: 
  $$\text{min\_samples} \ge D + 1$$
  where $D$ is the dimensionality of the dataset. Often, setting it to $2 \cdot D$ is recommended to make the clustering robust to noise.
- For noisy datasets, larger values (e.g., 10 or 20) help prevent spurious clusters from forming due to noise density spikes.

### Choosing $\epsilon$ (The K-Distance Plot)
We can calculate the distance from each point to its $k$-th nearest neighbor (where $k = \text{min\_samples} - 1$).
1. Compute these distances for all samples in the dataset.
2. Sort the distances in ascending order and plot them.
3. Look for the **"elbow"** (the point of maximum curvature). The elbow indicates the threshold where density drops off dramatically. Points before this elbow belong to dense clusters; points after are noise or border points. Set $\epsilon$ to the distance value at this elbow.

---

### Pros & Cons of DBSCAN
#### Pros:
- Does not require setting the number of clusters $K$ beforehand.
- Can find arbitrarily shaped clusters (e.g., nested rings, crescent moons).
- Built-in noise handling makes it robust to outliers.
#### Cons:
- **Varying Densities:** Performs poorly if clusters have highly varying densities. If $\epsilon$ is set for the sparse cluster, the dense clusters merge. If set for the dense cluster, the sparse cluster is labeled as noise.
- **Curse of Dimensionality:** In high-dimensional spaces, distance metrics become less informative, causing Euclidean distance to fail (all points appear distant and density is diluted).
- **Sensitivity to Parameters:** Small shifts in $\epsilon$ can drastically change the number of clusters or merge distinct clusters.

## 4. Code Setup & Imports

Let's prepare the Python environment and import the required libraries.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN, KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)
np.random.seed(42)

print("Libraries imported successfully!")

## 5. Demo A: DBSCAN Implementation From Scratch

We will build a custom class `DBSCANScratch` which identifies core points, border points, and noise. We will test it on a synthetic moons dataset and compare it directly to K-Means.

In [ ]:
class DBSCANScratch:
    def __init__(self, eps=0.5, min_samples=5):
        self.eps = eps
        self.min_samples = min_samples
        self.labels_ = None
        self.core_sample_indices_ = []
        self.components_ = None

    def fit(self, X):
        n_samples = X.shape[0]
        # -2: unvisited, -1: noise, >=0: cluster ID
        self.labels_ = np.full(n_samples, -2, dtype=int)
        self.core_sample_indices_ = []
        
        cluster_id = 0
        for idx in range(n_samples):
            if self.labels_[idx] != -2:
                continue
                
            # Find epsilon neighborhood of the current point
            neighbors = self._get_neighbors(X, idx)
            
            if len(neighbors) < self.min_samples:
                self.labels_[idx] = -1 # Label as noise temporarily
            else:
                self.core_sample_indices_.append(idx)
                self._expand_cluster(X, idx, neighbors, cluster_id)
                cluster_id += 1
                
        # Clean up core sample indices and components
        self.core_sample_indices_ = np.array(self.core_sample_indices_)
        if len(self.core_sample_indices_) > 0:
            self.components_ = X[self.core_sample_indices_]
        else:
            self.components_ = np.empty((0, X.shape[1]))
            
        return self

    def _get_neighbors(self, X, point_idx):
        # Vectorized Euclidean distance from query point to all other points
        distances = np.linalg.norm(X - X[point_idx], axis=1)
        return list(np.where(distances <= self.eps)[0])

    def _expand_cluster(self, X, start_idx, neighbors, cluster_id):
        self.labels_[start_idx] = cluster_id
        
        # Queue of neighbor indices to explore
        queue = list(neighbors)
        
        i = 0
        while i < len(queue):
            curr_idx = queue[i]
            
            # If previously labeled noise, it is a border point and becomes part of the cluster
            if self.labels_[curr_idx] == -1:
                self.labels_[curr_idx] = cluster_id
                
            # If unvisited
            elif self.labels_[curr_idx] == -2:
                self.labels_[curr_idx] = cluster_id
                
                # Check if this neighbor point is also a core point
                curr_neighbors = self._get_neighbors(X, curr_idx)
                if len(curr_neighbors) >= self.min_samples:
                    self.core_sample_indices_.append(curr_idx)
                    # Add new neighbor points to queue if they aren't already present
                    for neighbor in curr_neighbors:
                        if neighbor not in queue:
                            queue.append(neighbor)
            i += 1

# Generate crescent moons dataset
X_moons, y_moons = make_moons(n_samples=300, noise=0.08, random_state=42)

# Fit custom DBSCAN
db_scratch = DBSCANScratch(eps=0.2, min_samples=5)
db_scratch.fit(X_moons)
print(f"Scratch DBSCAN found {len(np.unique(db_scratch.labels_[db_scratch.labels_ >= 0]))} clusters.")
print(f"Number of Noise Points detected: {np.sum(db_scratch.labels_ == -1)}")

### DBSCAN vs K-Means Comparison
Let's visualize the clustering output of DBSCAN compared to K-Means on the crescent moon dataset.

In [ ]:
km = KMeans(n_clusters=2, random_state=42)
km_labels = km.fit_predict(X_moons)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot K-Means Results
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=km_labels, cmap='coolwarm', alpha=0.7, edgecolors='black')
axes[0].set_title("K-Means Clustering (K=2)\nFails on Non-Linear Geometries", fontsize=13, fontweight='bold')

# Plot DBSCAN Results
# Color noise points (label = -1) differently (black)
labels = db_scratch.labels_
unique_labels = np.unique(labels)
colors = ['#3498db', '#e74c3c']

for label in unique_labels:
    if label == -1:
        # Noise
        axes[1].scatter(X_moons[labels == label, 0], X_moons[labels == label, 1], 
                        color='black', marker='x', s=50, alpha=0.8, label='Noise')
    else:
        axes[1].scatter(X_moons[labels == label, 0], X_moons[labels == label, 1], 
                        color=colors[label % len(colors)], alpha=0.7, edgecolors='black', label=f'Cluster {label}')

axes[1].set_title("DBSCAN Clustering (eps=0.2, min_samples=5)\nCaptures Complex Shapes & Noise", fontsize=13, fontweight='bold')
axes[1].legend()
plt.tight_layout()
plt.show()

## 6. Demo B: Hyperparameter Sensitivity Visualization

Small variations in parameters dramatically alter results. Let's build a grid of plots visualizing how clustering behavior changes as we vary `eps` and `min_samples` on the same crescent moon dataset.

In [ ]:
eps_values = [0.1, 0.2, 0.4]
min_samples_values = [3, 7]

fig, axes = plt.subplots(len(min_samples_values), len(eps_values), figsize=(18, 11), sharex=True, sharey=True)

for r, ms in enumerate(min_samples_values):
    for c, eps in enumerate(eps_values):
        db = DBSCAN(eps=eps, min_samples=ms)
        labels = db.fit_predict(X_moons)
        
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = list(labels).count(-1)
        
        # Scatter plot
        ax = axes[r, c]
        scatter = ax.scatter(X_moons[:, 0], X_moons[:, 1], c=labels, cmap='tab10', alpha=0.6, edgecolors='none')
        
        # If there are noise points, plot them as black crosses
        if -1 in labels:
            ax.scatter(X_moons[labels == -1, 0], X_moons[labels == -1, 1], color='black', marker='x', s=40, alpha=0.8)
            
        ax.set_title(f"eps={eps}, min_samples={ms}\nClusters: {n_clusters} | Noise: {n_noise}", fontsize=11, fontweight='bold')

plt.suptitle("DBSCAN Parameter Sensitivity Grid", fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

## 7. Demo C: Auto-Tuning $\epsilon$ & Cluster Evaluation

To programmatically choose $\epsilon$, we use the **$k$-distance plot** method. Here, we set $k = \text{min\_samples}$ (e.g., 5), calculate the distance of each point to its $k$-th nearest neighbor, sort the distances, and look for the point of maximum curvature.

In [ ]:
# Step 1: Fit Nearest Neighbors
min_samples_k = 5
neighbors = NearestNeighbors(n_neighbors=min_samples_k)
neighbors.fit(X_moons)

# Step 2: Get distance matrix
distances, indices = neighbors.kneighbors(X_moons)

# Distance to the k-th nearest neighbor (last column)
k_distances = distances[:, -1]
k_distances.sort()

# Plot the k-distance curve
plt.figure(figsize=(10, 5))
plt.plot(k_distances, color='#8e44ad', linewidth=2.5)
plt.axhline(y=0.2, color='red', linestyle='--', alpha=0.8, label='Elbow threshold (~0.20)')
plt.title("5-Nearest Neighbor Distance Plot (K-Distance Plot)", fontsize=13, fontweight='bold')
plt.xlabel("Points sorted by distance")
plt.ylabel("5-NN Distance")
plt.legend()
plt.show()

### Clustering Validation Metrics
We can evaluate clustering results using metric models:
- **Adjusted Rand Index (ARI):** Measures cluster similarity against ground-truth labels, adjusted for chance (ranges from $-1$ to $1$; $1$ is perfect agreement).
- **Normalized Mutual Information (NMI):** Information-theoretic measure of mutual information between predicted clusters and true classes (ranges from $0$ to $1$).
- **Silhouette Score:** Distance-based metric (no ground truth required) assessing cohesion vs. separation (ranges from $-1$ to $1$).

In [ ]:
# Fit DBSCAN using optimized parameter
db_opt = DBSCAN(eps=0.2, min_samples=5)
opt_labels = db_opt.fit_predict(X_moons)

# Evaluate scores
ari = adjusted_rand_score(y_moons, opt_labels)
nmi = normalized_mutual_info_score(y_moons, opt_labels)
sil = silhouette_score(X_moons, opt_labels) if len(set(opt_labels)) > 1 else np.nan

print("DBSCAN Clustering Validation Metrics:")
print("=" * 50)
print(f"Adjusted Rand Index (ARI):          {ari:.4f}  (Ideal: 1.000)")
print(f"Normalized Mutual Info (NMI):       {nmi:.4f}  (Ideal: 1.000)")
print(f"Silhouette Score:                   {sil:.4f}  (No ground truth needed)")
print("=" * 50)

## Summary Checklist for DBSCAN

1. **Core Parameter Combination:**
   - `eps` ($\epsilon$): Defines region radius. Too small creates noise; too large merges clusters.
   - `min_samples`: Defines neighborhood density threshold. Larger values help filter out outlier noise.
2. **Point Classifications:**
   - *Core:* Contains $\ge$ `min_samples` points in $\epsilon$-neighborhood.
   - *Border:* Located in the neighborhood of a core point but contains fewer than `min_samples` points.
   - *Noise:* Neither core nor border.
3. **Density Connectivity & Reachability:** Clusters are defined as dense zones composed of density-connected Core points and their border points.
4. **Parameter Selection Heuristics:**
   - *min_samples:* Set $\ge D + 1$ (usually $2 \cdot D$).
   - *eps:* Find the elbow point in the sorted $k$-nearest neighbor distance plot.
5. **Constraints & Drawbacks:** Struggles with varying density levels, high dimensions (curse of dimensionality), and is highly sensitive to distance scaling.